# Relax Take Home Challenge

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [6]:
DATA_DIR = Path('/Users/justinku/Documents/GitHub/JustinkSpringboard/relax_challenge')
users = pd.read_csv(DATA_DIR / "takehome_users.csv", encoding="latin1")         
eng   = pd.read_csv(DATA_DIR / "takehome_user_engagement.csv", encoding="latin1")

In [7]:
# parsing dates and keys

users.columns = [c.lower() for c in users.columns]
eng.columns   = [c.lower() for c in eng.columns]

to_dt = lambda s: pd.to_datetime(s, errors="coerce")
users["creation_dt"] = to_dt(users["creation_time"])
eng["login_dt"] = to_dt(eng.get("time_stamp", eng.get("visited_at")))

eng["user_key"]   = eng.get("user_id", eng.filter(like="user").filter(like="id").iloc[:,0]).astype(str)
users["user_key"] = users["object_id"].astype(str)

In [8]:
# labeling adopted 
    # 3 or more logins in a 7-day period

eng = eng.dropna(subset=["login_dt"]).copy()
eng["login_day"] = eng["login_dt"].dt.normalize()

def adopted_from_days(days: pd.Series) -> bool:
    d = np.array(sorted(days.unique()))
    i = 0
    for j in range(len(d)):
        while d[j] - d[i] > np.timedelta64(6, "D"):  # 7-day inclusive window
            i += 1
        if (j - i + 1) >= 3:
            return True
    return False

adopted = (eng.groupby("user_key")["login_day"]
             .apply(adopted_from_days)
             .rename("adopted")
             .to_frame())

In [9]:
# joining to users

df = users.merge(adopted, on="user_key", how="left")
df["adopted"] = df["adopted"].fillna(False)

/var/folders/j4/zp2fvcv16r3_602nppyngwfc0000gn/T/ipykernel_46961/1888614044.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["adopted"] = df["adopted"].fillna(False)


In [10]:
print("Overall adoption rate:", f"{df['adopted'].mean():.1%}")
if "creation_source" in df.columns:
    print("\nAdoption by creation_source:")
    print(df.groupby("creation_source")["adopted"].mean().sort_values(ascending=False).round(3))

Overall adoption rate: 13.4%

Adoption by creation_source:
creation_source
SIGNUP_GOOGLE_AUTH    0.168
GUEST_INVITE          0.166
SIGNUP                0.140
ORG_INVITE            0.130
PERSONAL_PROJECTS     0.078
Name: adopted, dtype: float64
